In [ ]:
# load the names dataset from file
from pathlib import Path

data_dir = Path.cwd() / ".." / ".." / "data"


def load_names(path: Path) -> list[str]:
    with path.open("r") as f:
        return f.read().splitlines()


words = load_names(data_dir / "names.txt")
words[:4]

**Create the Training Set**

In [ ]:
# global sentinel tokens (start and stop)
TOKEN_DOT = "."
# the vocabulary size
VOCAB_SIZE = 27

In [ ]:
# LUT construction
chars = sorted(list(set("".join(words))))

# string-to-index
stoi = {c: i + 1 for i, c in enumerate(chars)}
stoi[TOKEN_DOT] = 0

# index to string
itos = {i: c for c, i in stoi.items()}

assert len(stoi) == len(itos), "broken invariant"
assert all(itos[stoi[c]] == c for c in chars), "broken invariant"

In [ ]:
# create the training set (all bigrams)
import torch

xs_raw, ys_raw = [], []
for w in words[:1]:
    chs = [TOKEN_DOT] + list(w) + [TOKEN_DOT]
    for l, r in zip(chs, chs[1:]):
        xs_raw.append(stoi[l])
        ys_raw.append(stoi[r])

xs = torch.tensor(xs_raw)
ys = torch.tensor(ys_raw)

In [ ]:
# extract the bigrams provided by a single word
def bigrams(s: str) -> list[tuple[str, str]]:
    chs = [TOKEN_DOT] + list(s) + [TOKEN_DOT]
    grams = []
    for l, r in zip(chs, chs[1:]):
        grams.append((l, r))
    return grams


for bg in bigrams(words[0]):
    print(bg)

**One-Hot Encoding**

In [ ]:
import torch.nn.functional as F

xenc = F.one_hot(xs, num_classes=VOCAB_SIZE).float()

**Constructing the Network**

In [ ]:
# initialize the network
W = torch.randn((VOCAB_SIZE, VOCAB_SIZE))

In [ ]:
# matmul with one-hot encoded input is just row selection

# lets lookup the row in W that corresponds to input token 'e'
index = stoi["e"]
print(W[index])

# second example is ('e', 'm'), so xenc[1] encodes 'e'
print(xenc[1] @ W)

**Aside: Exponentiation**

In [ ]:
# we interpret the output of the single linear layer as the "log counts"
# for the character (conditioned on the previous one); we exponentiate to get counts
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-3, 3, num=100)
y = np.exp(x)

plt.plot(x, y)
plt.axvline()
plt.title("The Exponential Function (-3 to 3)")
plt.xlabel("x")
plt.ylabel("f(x)")

**Transforming Outputs**

In [ ]:
logits = xenc @ W  # log counts
counts = logits.exp()  # equivalent to the N array from previous implementation
probs = counts / counts.sum(
    axis=1, keepdim=True
)  # normalize to probability distribution, as before

**Vectorized Loss**

In [ ]:
loss = -probs[torch.arange(5), ys].log().mean()
loss

**Complete Forward Pass and Walkthrough**

In [ ]:
# initialize the network
gen = torch.Generator()
gen.manual_seed(1337 + 1)

W = torch.randn((VOCAB_SIZE, VOCAB_SIZE), generator=gen)

In [ ]:
# forward
counts = (xenc @ W).exp()
probs = counts / counts.sum(axis=1, keepdim=True)

In [ ]:
for i in range(xs.nelement()):
    l = itos[xs[i].item()]
    r = itos[ys[i].item()]

    print(f"bigram is ({l},{r})")
    print(f"  {l} index = {xs[i]}")
    print(f"  {r} index = {ys[i]}")

    prob = probs[i, ys[i]]
    print(f"  probability assigned = {prob.item() * 100:.2f}%")
    print(f"  loss = {prob.log():.2f}")

# aggregate loss
loss = -probs[torch.arange(xs.nelement()), ys].log().mean()
print(f"aggregate loss = {loss:.3f}")

**Backward Pass & Update**

In [ ]:
# initialize the network
gen = torch.Generator()
gen.manual_seed(1337)

W = torch.randn((VOCAB_SIZE, VOCAB_SIZE), generator=gen, requires_grad=True)

In [ ]:
# forward
counts = (xenc @ W).exp()
probs = counts / counts.sum(axis=1, keepdim=True)

In [ ]:
loss = -probs[torch.arange(xs.nelement()), ys].log().mean()
print(f"loss = {loss:.3f}")

In [ ]:
W.grad = None  # zero the gradient
loss.backward()

In [ ]:
# update
W.data += -0.1 * W.grad

**Putting it Together**

In [ ]:
import torch

xs_raw, ys_raw = [], []
for w in words:
    chs = [TOKEN_DOT] + list(w) + [TOKEN_DOT]
    for l, r in zip(chs, chs[1:]):
        xs_raw.append(stoi[l])
        ys_raw.append(stoi[r])

xs = torch.tensor(xs_raw)
ys = torch.tensor(ys_raw)

In [ ]:
# initialize the network
gen = torch.Generator()
gen.manual_seed(1337)

W = torch.randn((VOCAB_SIZE, VOCAB_SIZE), requires_grad=True, generator=gen)

In [ ]:
import torch.nn.functional as F

for k in range(1024):
    # encoding
    xenc = F.one_hot(xs, num_classes=VOCAB_SIZE).float()

    # forward pass
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(axis=1, keepdim=True)

    # loss
    loss = -probs[torch.arange(xs.nelement()), ys].log().mean()
    if k % 100 == 0:
        print(f"[iteration {k}] loss = {loss:.2f}")

    # backward pass
    W.grad = None
    loss.backward()

    # update
    lr = 50.0 * (0.99**k)
    W.data += -lr * W.grad

# final training loss
print(loss.item())

**Sampling from the Model**

In [ ]:
def sample_one(model: torch.Tensor, g: torch.Generator) -> str:
    """Sample a single word from the model."""
    word = ""

    ix = 0  # 0 is the index of the start token '.'
    while True:
        # forward pass
        input = (
            F.one_hot(torch.tensor(ix), num_classes=VOCAB_SIZE)
            .reshape((1, VOCAB_SIZE))
            .float()
        )
        counts = (input @ model).exp()
        probs = counts / counts.sum(axis=1, keepdim=True)

        # sample an index from the distribution
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()

        # check if this is the stop token
        if ix == 0:
            return word

        # add the character to the growing word
        word += itos[ix]

In [ ]:
gen = torch.Generator()
gen.manual_seed(1337)

for _ in range(5):
    print(sample_one(W, gen))

**Packaging things Up**

In [ ]:
from makemore.vocab import Vocab
from makemore.bigram_nn import BigramNN
from makemore.train import SGD, train

vocab = Vocab.from_words(words)

model = BigramNN(vocab)
opt = SGD(model.parameters(), lr=50.0)
history = train(
    model, words, opt, steps=1024, lr_schedule=lambda i: 50.0 * (0.99**i)
)

In [ ]:
print(f"loss = {history[-1]}")

In [ ]:
for name in model.sample(5):
    print(name)